This Jupyter notebook was my first attempt at defining how my dataset will look like. The official Python file will be based on this one

In [73]:
#Import dependencies

import json
import re
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_JSON = Path("test_warnings.json")
OUTPUT_DIR = Path("trying_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [74]:
#Helper functions

def safe_column_name(name):
    """Convert API names into consistent column names."""
    name = str(name).strip().lower()
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_")


def get_weather_measurements(weather):
    """
    Flatten MountainWeather.MeasurementTypes into columns.

    Example:
    Wind -> Speed -> Fresh breeze
    becomes:
    weather_wind_speed = 'Fresh breeze'
    """
    result = {}

    for measurement in weather.get("MeasurementTypes", []) or []:
        measurement_name = measurement.get("Name", "unknown")

        for subtype in measurement.get("MeasurementSubTypes", []) or []:
            subtype_name = subtype.get("Name", "unknown")

            column = (
                "weather_"
                + safe_column_name(measurement_name)
                + "_"
                + safe_column_name(subtype_name)
            )

            result[column] = subtype.get("Value")

    return result

In [75]:
# Load JSON

with INPUT_JSON.open("r", encoding="utf-8") as file:
    warnings = json.load(file)

if not isinstance(warnings, list):
    raise ValueError("Expected a JSON list of warning records.")

In [76]:
#List variables

metadata_columns = [
    "RegId",
    "RegionId",
    "RegionName",
    "RegionTypeName",
    "DangerLevel",
    "ValidFrom",
    "ValidTo",
    "PublishTime",
]

time_columns = [
    "wind_shift_start",
    "wind_shift_end",
    "freezing_level_start",
    "freezing_level_end",
    "rain_snow_line_start",
    "rain_snow_line_end",
]

numeric_columns = [
    "DangerLevel",
    "rainfall_most_exposed_area",
    "rainfall_average",
    "temperature_min",
    "temperature_max",
    "freezing_level_masl",
    "rain_snow_line_altitude_masl",
]

In [77]:
# Construct columns for dataset

rows = []

for warning in warnings:
    #Create initial dictionary with top-level metadata field
    row = {field: warning.get(field) for field in metadata_columns}

    weather = warning.get("MountainWeather") or {}

    # Cloud cover.
    row["weather_cloud_cover"] = weather.get("CloudCoverName")

    # Flatten weather measurements.
    measurements = get_weather_measurements(weather)

    # Rainfall.
    row["rainfall_most_exposed_area"] = measurements.get("weather_rainfall_most_exposed_area")
    row["rainfall_average"] = measurements.get("weather_rainfall_average")

    # Initial wind speed.
    row["wind_speed"] = measurements.get("weather_wind_speed")
    row["wind_direction"] = measurements.get("weather_wind_direction")

    # Temperature.
    row["temperature_min"] = measurements.get("weather_temperature_min")
    row["temperature_max"] = measurements.get("weather_temperature_max")

    # Freezing level.
    row["freezing_level_masl"] = measurements.get("weather_freezing_level_masl")
    row["freezing_level_start"] = measurements.get("weather_freezing_level_timeperiode_start")
    row["freezing_level_end"] = measurements.get("weather_freezing_level_timeperiode_end")

    # Rain/snow line.
    row["rain_snow_line_altitude_masl"] = measurements.get("weather_rain_snow_line_altitude_masl")

    row["rain_snow_line_start"] = measurements.get("weather_rain_snow_line_altitude_timeperiode_start")
    row["rain_snow_line_end"] = measurements.get("weather_rain_snow_line_altitude_timeperiode_end")

    # Wind-shift
    row["wind_shift_speed"] = measurements.get("weather_wind_shift_to_speed")
    row["wind_shift_direction"] = measurements.get("weather_wind_shift_to_direction")

    # Wind-shift time endpoints.
    row["wind_shift_start"] = measurements.get("weather_wind_shift_to_timeperiode_start")
    row["wind_shift_end"] = measurements.get("weather_wind_shift_to_timeperiode_end")

    rows.append(row)

df = pd.DataFrame(rows)

In [78]:
# Derive features for published time

df["publish_month"] = pd.to_datetime(df["PublishTime"],errors="coerce").dt.month
df["publish_hour"] = pd.to_datetime(df["PublishTime"],errors="coerce").dt.hour

In [79]:
# Convert time endpoints to numeric values

for column in time_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Replace null values with zero:
df[time_columns] = df[time_columns].fillna(0)

In [80]:
#Convert numeric columns to actual numbers
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[numeric_columns] = df[numeric_columns].fillna(0)

In [81]:
#If no wind change -> fill wind_shift with prevailing wind
df["wind_shift_speed"] = df["wind_shift_speed"].fillna(df["wind_speed"])
df["wind_shift_direction"] = df["wind_shift_direction"].fillna(df["wind_direction"])

#Quantify wind direction change
direction_degrees = {
    "N": 0,
    "NE": 45,
    "E": 90,
    "SE": 135,
    "S": 180,
    "SW": 225,
    "W": 270,
    "NW": 315,
}

df["wind_direction_degrees"] = (df["wind_direction"].map(direction_degrees))

df["wind_shift_direction_degrees"] = (df["wind_shift_direction"].map(direction_degrees))

# Smallest angular change between directions.
difference = (df["wind_shift_direction_degrees"]- df["wind_direction_degrees"]).abs()

df["wind_direction_change"] = np.minimum(difference,360 - difference)

In [82]:
geographical_region_by_id = {
    # Svalbard / Arctic
    3001: "Svalbard / Arctic",
    3002: "Svalbard / Arctic",
    3003: "Svalbard / Arctic",
    3004: "Svalbard / Arctic",

    # Northern Norway
    3005: "Northern Norway",  # Øst-Finnmark
    3006: "Northern Norway",  # Finnmarkskysten
    3007: "Northern Norway",  # Vest-Finnmark
    3008: "Northern Norway",  # Finnmarksvidda
    3009: "Northern Norway",  # Nord-Troms
    3010: "Northern Norway",  # Lyngen
    3011: "Northern Norway",  # Tromsø
    3012: "Northern Norway",  # Sør-Troms
    3013: "Northern Norway",  # Indre Troms
    3014: "Northern Norway",  # Lofoten og Vesterålen
    3015: "Northern Norway",  # Ofoten
    3016: "Northern Norway",  # Salten
    3017: "Northern Norway",  # Svartisen
    3018: "Northern Norway",  # Helgeland

    # Trøndelag
    3019: "Trøndelag",  # Nord-Trøndelag
    3020: "Trøndelag",  # Sør-Trøndelag

    # Western Norway
    3021: "Western Norway",  # Ytre Nordmøre
    3022: "Western Norway",  # Trollheimen
    3023: "Western Norway",  # Romsdal
    3024: "Western Norway",  # Sunnmøre
    3026: "Western Norway",  # Ytre Fjordane
    3027: "Western Norway",  # Indre Fjordane
    3028: "Western Norway",  # Jotunheimen
    3029: "Western Norway",  # Indre Sogn
    3030: "Western Norway",  # Ytre Sogn
    3031: "Western Norway",  # Voss
    3033: "Western Norway",  # Hordalandskysten
    3034: "Western Norway",  # Hardanger
    3036: "Western Norway",  # Rogalandskysten

    # Southern Norway
    3037: "Southern Norway",  # Heiane
    3038: "Southern Norway",  # Agder sør

    # Eastern Norway
    3025: "Eastern Norway",  # Nord-Gudbrandsdalen
    3032: "Eastern Norway",  # Hallingdal
    3035: "Eastern Norway",  # Vest-Telemark
    3039: "Eastern Norway",  # Telemark sør
    3040: "Eastern Norway",  # Vestfold
    3041: "Eastern Norway",  # Buskerud sør
    3042: "Eastern Norway",  # Oppland sør
    3043: "Eastern Norway",  # Hedmark
    3044: "Eastern Norway",  # Akershus
    3045: "Eastern Norway",  # Oslo
    3046: "Eastern Norway",  # Østfold
}

df["geographical_region"] = (df["RegionId"].map(geographical_region_by_id))

In [83]:
#  ORGANIZE COLUMNS

column_order = [
    # Identifiers and target
    "RegId",
    "RegionId",
    "RegionName",
    "geographical_region",
    "RegionTypeName",
    "DangerLevel",

    # Warning timing
    "ValidFrom",
    "ValidTo",
    "PublishTime",
    "publish_hour",
    "publish_month",

    # Weather predictors
    "weather_cloud_cover",
    "rainfall_most_exposed_area",
    "rainfall_average",
    "wind_speed",
    "wind_shift_speed",
    "wind_direction",
    "wind_shift_direction",
    "wind_direction_change",
    "wind_shift_start",
    "wind_shift_end",
    "temperature_min",
    "temperature_max",
    "freezing_level_masl",
    "freezing_level_start",
    "freezing_level_end",
    "rain_snow_line_altitude_masl",
    "rain_snow_line_start",
    "rain_snow_line_end",
]

df = df[column_order]

In [84]:
output_csv = OUTPUT_DIR / "warnings.csv"
df.to_csv(output_csv,index=False,encoding="utf-8-sig")
print("\nSaved!")


Saved!
